# Diet, Sex, and Microbiome Structure in Aging: A Multi-View Analysis of the NUAGE Cohort


## Dataset and Scope

- Cohort: NUAGE
- Samples: 1,220 microbiome profiles
- Subjects: 610 individuals measured at two time points (`T0` and `T1`)
- Taxonomic resolution: species-level abundance profiles
- Metadata coverage: inflammatory markers, cognition, frailty and strength measures, age, gender, polypharmacy, microbiome score, and Mediterranean diet adherence score

This project integrates three analyses:

1. Mediation modeling to test whether microbiome-score changes mediate diet-associated changes in health outcomes.
2. Baseline gender effect-size estimation across five clinically relevant metrics.
3. Unsupervised clustering of microbiome profiles and testing cluster associations with clinical variables.

## 1. Mediation Analysis: Diet -> Microbiome -> Health Outcomes

The first analysis follows the original R workflow using change scores from `T0` to `T1`:

- `delta_food`
- `delta_microbiome`
- `delta_hsCRP`
- `delta_cspraxis`
- `delta_hgtdommean`

Each model adjusts for `age_t0`, `gender_t0`, and `PolyPharmacy_t0`. The indirect effect is represented by the ACME, which captures whether diet-associated change passes through microbiome-score change before reaching the outcome.

### Reproduced Mediation Summary
| Outcome | N | ACME | CI_low | CI_high | p_boot | Significant |
| --- | --- | --- | --- | --- | --- | --- |
| hsCRP (log delta) | 603 | -0.000421 | -0.001637 | 0.000025 | 0.086 | No |
| cspraxis (delta) | 610 | -0.000307 | -0.002969 | 0.002356 | 0.849 | No |
| hgtdommean (delta) | 610 | -0.001146 | -0.007032 | 0.002361 | 0.543 | No |

### Interpretation

- None of the three reproduced indirect effects crossed the usual significance threshold.
- The ACME confidence intervals for hsCRP, cspraxis, and hgtdommean all included zero.
- In this cohort, the provided workflow did not support strong evidence that microbiome-score change mediated the relationship between diet-score change and the three selected outcomes after covariate adjustment.

![Mediation forest plot](mediation_acme_forest_plot.png)

In [ ]:
import numpy as np
import pandas as pd

metadata = pd.read_csv("NUAGE_Metadata.csv")
id_col = metadata.columns[0]

t0 = metadata[metadata[id_col].astype(str).str.endswith("_T0")].copy()
t1 = metadata[metadata[id_col].astype(str).str.endswith("_T1")].copy()
t0[id_col] = t0[id_col].str.replace("_T0", "", regex=False)
t1[id_col] = t1[id_col].str.replace("_T1", "", regex=False)

merged = pd.merge(t0, t1, on=id_col, suffixes=("_t0", "_t1"))
merged["delta_food"] = merged["food_scores_t1"] - merged["food_scores_t0"]
merged["delta_microbiome"] = merged["microbiome_scores_t1"] - merged["microbiome_scores_t0"]
merged["delta_hsCRP"] = merged["hsCRP_t1"] - merged["hsCRP_t0"]
merged["delta_cspraxis"] = merged["cspraxis_t1"] - merged["cspraxis_t0"]
merged["delta_hgtdommean"] = merged["hgtdommean_t1"] - merged["hgtdommean_t0"]
merged["delta_hsCRP_log"] = np.log1p(merged["delta_hsCRP"] - merged["delta_hsCRP"].min())

# The original assignment used the R mediation package.
# This project notebook reproduces the same delta-score design:
# delta_microbiome ~ delta_food + age_t0 + gender_t0 + PolyPharmacy_t0
# delta_outcome ~ delta_food + delta_microbiome + age_t0 + gender_t0 + PolyPharmacy_t0

## 2. Baseline Gender Differences with Hedges' g

The second analysis estimates standardized male-versus-female differences at baseline (`T0`) across:

- `leptin`
- `hsCRP`
- `hgtdommean`
- `microbiome_scores`
- `food_scores`

Hedges' g was used instead of Cohen's d because the baseline sex groups were slightly unbalanced, making the small-sample correction appropriate.

### Effect Size Summary
| Metric | Male_n | Female_n | Hedges_g | CI_low | CI_high | p_value |
| --- | --- | --- | --- | --- | --- | --- |
| leptin | 282 | 321 | -1.007 | -1.177 | -0.837 | 0.0000 |
| hsCRP | 282 | 321 | 0.053 | -0.107 | 0.213 | 0.5176 |
| hgtdommean | 284 | 326 | 2.383 | 2.176 | 2.591 | 0.0000 |
| microbiome_scores | 284 | 326 | -0.035 | -0.193 | 0.124 | 0.6751 |
| food_scores | 284 | 326 | -0.057 | -0.216 | 0.102 | 0.4780 |

### Interpretation

- `hgtdommean` showed a very large positive effect size, indicating much higher baseline grip strength in males.
- `leptin` showed a large negative effect size, indicating higher baseline leptin levels in females.
- `hsCRP`, `microbiome_scores`, and `food_scores` had near-zero effect sizes with confidence intervals spanning zero, suggesting minimal baseline gender differences in those measures.

![Gender effect sizes](gender_effect_size_forest_plot.png)

In [ ]:
import math
import pandas as pd

metadata = pd.read_csv("NUAGE_Metadata.csv")
id_col = metadata.columns[0]
t0 = metadata[metadata[id_col].astype(str).str.endswith("_T0")].copy()

def hedges_g(male_values, female_values):
    n1, n2 = len(male_values), len(female_values)
    m1, m2 = male_values.mean(), female_values.mean()
    v1, v2 = male_values.var(ddof=1), female_values.var(ddof=1)
    pooled = ((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2)
    d_value = (m1 - m2) / math.sqrt(pooled)
    correction = 1 - (3 / (4 * (n1 + n2 - 2) - 1))
    g_value = d_value * correction
    var_d = (n1 + n2) / (n1 * n2) + (g_value ** 2) / (2 * (n1 + n2 - 2))
    var_g = (correction ** 2) * var_d
    se = math.sqrt(var_g)
    return g_value, g_value - 1.96 * se, g_value + 1.96 * se

## 3. Microbiome Clustering and Clinical Association Testing

The third analysis used unsupervised learning on the full species abundance table:

- `log1p` transformation to reduce the influence of extreme counts
- z-score standardization across species
- PCA retaining 90% of the variance
- K-means clustering tested for `k = 2` through `k = 10`
- Silhouette score used to choose the optimal number of clusters

### Key Results

- Raw species features before PCA: `1,896`
- PCA components retained at 90% variance: `194`
- Optimal cluster count: `k = 2`
- Best silhouette score: approximately `0.25`

### Clinical Association Results

- `hsCRP`: not significant (`p = 0.332`)
- `cspraxis`: significant (`p = 0.005`)
- `hgtdommean`: not significant (`p = 0.078`)

### Interpretation

The cohort separates into two microbiome-profile clusters under the provided preprocessing pipeline. These clusters were not strongly associated with baseline inflammation or grip strength, but they did show a statistically meaningful difference in constructional praxis scores, suggesting a link between microbiome structure and one dimension of cognitive performance.

![Optimal k](optimal_k.png)

![PCA clustering](pca_clustering_scatterplot.png)

![Clinical associations](binary_clinical_associations_boxplots.png)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

species_df = pd.read_csv("NUAGE_SpProfile.csv", index_col=0)
metadata_df = pd.read_csv("NUAGE_Metadata.csv", index_col=0)
common_samples = species_df.index.intersection(metadata_df.index)
species_df = species_df.loc[common_samples]
metadata_df = metadata_df.loc[common_samples]

species_log = np.log1p(species_df)
species_scaled = StandardScaler().fit_transform(species_log)
species_pca = PCA(n_components=0.90, random_state=42).fit_transform(species_scaled)

scores = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(species_pca)
    scores.append((k, silhouette_score(species_pca, labels)))

## Final Project Conclusion

Diet, Sex, and Microbiome Structure in Aging: A Multi-View Analysis of the NUAGE Cohort presents Assignment 2 as a coherent microbiome analytics project rather than a set of isolated tasks. Across longitudinal mediation modeling, baseline effect-size estimation, and unsupervised clustering, the NUAGE cohort revealed three main insights:

- baseline sex differences were strongest for leptin and grip strength,
- microbiome-score change did not show strong mediation evidence for the selected health outcomes,
- microbiome-profile clusters were most clearly associated with cognitive function rather than with inflammation or grip strength.